# NISAR, BIOMASS & AGBD Extraction  

**Date:** March 2nd,2026

**Author**:Harshini Girish(UAH), Alex Mandel(Development Seed)

**Description**:This notebook checks whether our AOI is covered by the public catalogs for **NISAR** (via CMR STAC/ASF) and **ESA BIOMASS** (via the ESA MAAP STAC catalogue), and then uses **GEDI L4A AGBD** (footprint-level aboveground biomass density) to extract biomass samples near the AOI/sites. We load the site geometry (KML/GeoJSON), compute the AOI bounding box (with an optional buffer for discovery), run STAC searches to confirm coverage and visualize footprints, then download a small set of GEDI granules and extract quality-filtered AGBD shots. Finally, we spatially join GEDI shots to each site (optionally using a configurable meter-based buffer when sites are small or GEDI sampling is sparse) and produce a per-site summary table (count, mean/median, quantiles, min/max) that can be exported for downstream analysis and fusion planning. 


## Run This Notebook

To access and run this tutorial within MAAP’s Algorithm Development Environment (ADE), please refer to the [Getting started with the MAAP](#) section of our documentation.

**Disclaimer**: It is highly recommended to run this tutorial within MAAP’s ADE, which already includes packages specific to MAAP, such as maap-py. Running the tutorial outside of the MAAP ADE may lead to errors.


## Install/Import Packages
 
Let's install and load the packages necessary for this tutorial.

In [7]:
import json
from shapely.geometry import box, mapping, Point
import pandas as pd
import geopandas as gpd
import folium
import pystac_client
import earthaccess
import h5py
import numpy as np
from pystac import ItemCollection

## Inputs  
Edit these paths/parameters to match your AOI and the collections you want to check.

In [3]:
# --- AOI / Sites input (KML/GeoJSON/Shapefile/etc.) ---
SITES_PATH = "amacayacu_100m.kml"   # change if needed
SITE_ID_COL = None  # set to an existing column name, or leave None to auto-create site_id

# --- STAC endpoints ---
NISAR_STAC_URL   = "https://cmr.earthdata.nasa.gov/stac/ASF"
BIOMASS_STAC_URL = "https://catalog.maap.eo.esa.int/catalogue/"

# --- Collections (use exact STAC collection IDs) ---
NISAR_COLLECTION = "NISAR_L2_GCOV_BETA_V1_1"
BIOMASS_COLLECTIONS = ["BiomassLevel1b"]

# --- Optional datetime filters (ISO8601 interval), or None ---
NISAR_DT = None
BIOMASS_DT = None

# --- GEDI AGBD dataset (from dataset discovery) ---
GEDI_SHORT = "GEDI_L4A_AGB_Density_V2_1_2056"

# Use a buffer for GEDI discovery/extraction (degrees). Keep modest; increase if needed.
BBOX_BUFFER_DEG = 0.25

# Buffer radius for "site neighborhood" summaries (meters). Set 0 for strict in-polygon only.
SITE_BUFFER_M = 50_000


## Load AOI / sites and compute bbox  
We read the sites file into a GeoDataFrame (EPSG:4326), ensure a `site_id`, and compute the AOI bbox for catalog searches.

In [6]:
sites_gdf = gpd.read_file(SITES_PATH)

# Ensure WGS84 lon/lat
if sites_gdf.crs is None:
    sites_gdf = sites_gdf.set_crs("EPSG:4326")
else:
    sites_gdf = sites_gdf.to_crs("EPSG:4326")

# Ensure a site_id column
if SITE_ID_COL and SITE_ID_COL in sites_gdf.columns:
    sites_gdf = sites_gdf.rename(columns={SITE_ID_COL: "site_id"})
elif "site_id" not in sites_gdf.columns:
    sites_gdf["site_id"] = [f"site_{i+1}" for i in range(len(sites_gdf))]

aoi_geom = sites_gdf.geometry.union_all()
minx, miny, maxx, maxy = aoi_geom.bounds
BBOX = (minx, miny, maxx, maxy)

# buffered bbox for discovery
buf = BBOX_BUFFER_DEG
BBOX_BUF = (minx-buf, miny-buf, maxx+buf, maxy+buf)

print("AOI bbox:", BBOX)
print("Buffered bbox:", BBOX_BUF)
sites_gdf[["site_id", "geometry"]].head()


AOI bbox: (-70.2926180592577, -3.81826995097037, -70.2404260082027, -3.78652050791343)
Buffered bbox: (-70.5426180592577, -4.06826995097037, -69.9904260082027, -3.53652050791343)


,site_id,geometry
0,site_1,"MULTIPOLYGON (((-70.29262 -3.7921, -70.29262 -..."


## STAC helpers  
- `stac_search`: runs a bbox+collection STAC search  
- `items_to_featurecollection`: converts STAC Items directly to a GeoJSON FeatureCollection 

In [8]:
def stac_search(catalog_url, collections, bbox, datetime=None, limit=200):
    """Run a STAC bbox search and return a PySTAC ItemCollection."""
    cat = pystac_client.Client.open(catalog_url)
    search = cat.search(collections=collections, bbox=bbox, datetime=datetime, max_items=limit)
    items = list(search.items())
    return ItemCollection(items)

def stac_items_to_gdf(items):
    """Convert a PySTAC ItemCollection to a GeoDataFrame (EPSG:4326)."""
    return gpd.GeoDataFrame.from_features(items.to_dict(), crs="EPSG:4326")


## NISAR coverage 
Searches the NISAR STAC catalog for the specified collection intersecting the AOI bbox.

In [9]:
nisar_items = stac_search(
    catalog_url=NISAR_STAC_URL,
    collections=[NISAR_COLLECTION],
    bbox=BBOX,
    datetime=NISAR_DT,
    limit=500
)
print(f"NISAR items found for {NISAR_COLLECTION}: {len(nisar_items)}")


NISAR items found for NISAR_L2_GCOV_BETA_V1_1: 9


## BIOMASS coverage 
Searches the BIOMASS STAC catalog for the selected BIOMASS collections intersecting the AOI bbox.

In [11]:
biomass_items = stac_search(
    catalog_url=BIOMASS_STAC_URL,
    collections=BIOMASS_COLLECTIONS,
    bbox=BBOX,
    datetime=BIOMASS_DT,
    limit=500
)
print(f"BIOMASS items found ({BIOMASS_COLLECTIONS}): {len(biomass_items)}")

BIOMASS items found (['BiomassLevel1b']): 7


## Quick map: AOI + catalog footprints  
This creates a Folium map with:
- AOI polygon(s)
- NISAR item footprints (if any)
- BIOMASS item footprints (if any)

In [12]:
m = folium.Map(location=[(miny+maxy)/2, (minx+maxx)/2], zoom_start=8, tiles="OpenStreetMap")

# AOI
folium.GeoJson(
    data={"type":"Feature","geometry":mapping(aoi_geom),"properties":{"name":"AOI"}},
    name="AOI",
).add_to(m)

# NISAR footprints (GeoDataFrame.from_features(items.to_dict()))
if len(nisar_items) > 0:
    nisar_gdf = stac_items_to_gdf(nisar_items)
    folium.GeoJson(nisar_gdf.__geo_interface__, name="NISAR").add_to(m)

# BIOMASS footprints (GeoDataFrame.from_features(items.to_dict()))
if len(biomass_items) > 0:
    biomass_gdf = stac_items_to_gdf(biomass_items)
    folium.GeoJson(biomass_gdf.__geo_interface__, name="BIOMASS").add_to(m)

folium.LayerControl().add_to(m)
m

##  GEDI L4A AGBD: search and download granules  
We use `earthaccess` to search for GEDI L4A AGBD granules intersecting the **buffered bbox** (to avoid missing sparse tracks). Then we download a small batch first; increase the slice once you’re happy with results.

In [13]:
earthaccess.login()

gedi_results = earthaccess.search_data(
    short_name=GEDI_SHORT,
    bounding_box=BBOX_BUF,
    count=200
)
print("GEDI granules found:", len(gedi_results))

# Start small; increase once validated
to_get = gedi_results[:10]
downloaded = earthaccess.download(to_get, local_path="gedi_l4a_agbd")
print("Downloaded files:", len(downloaded))
downloaded[:2]

GEDI granules found: 138


QUEUEING TASKS | :   0%|          | 0/10 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/10 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/10 [00:00<?, ?it/s]

Downloaded files: 10


[PosixPath('gedi_l4a_agbd/GEDI04_A_2019110092939_O01996_01_T03334_02_002_02_V002.h5'),
 PosixPath('gedi_l4a_agbd/GEDI04_A_2019137101716_O02416_04_T03984_02_002_02_V002.h5')]

## Extract AOI-filtered GEDI AGBD shots  
This reads each downloaded GEDI HDF5 file and extracts **shot-level**:
- `agbd`
- `lat_lowestmode`, `lon_lowestmode`
- quality flags (`l4_quality_flag`, `degrade_flag`)
…and filters to:
- valid AGBD (not fill)
- within the **buffered AOI bbox** (fast spatial filter)

In [14]:
AOI_MINX, AOI_MINY, AOI_MAXX, AOI_MAXY = BBOX_BUF

def extract_l4a_agbd_points_aoi(h5_path):
    rows = []
    with h5py.File(h5_path, "r") as f:
        beams = [k for k in f.keys() if k.startswith("BEAM")]
        for beam in beams:
            g = f[beam]
            if not all(k in g for k in ["agbd", "lat_lowestmode", "lon_lowestmode"]):
                continue

            agbd = g["agbd"][:]
            lat  = g["lat_lowestmode"][:]
            lon  = g["lon_lowestmode"][:]

            q4 = g["l4_quality_flag"][:] if "l4_quality_flag" in g else None
            degr = g["degrade_flag"][:] if "degrade_flag" in g else None

            m = (agbd > -9000) & np.isfinite(agbd) & np.isfinite(lat) & np.isfinite(lon)
            m &= (lon >= AOI_MINX) & (lon <= AOI_MAXX) & (lat >= AOI_MINY) & (lat <= AOI_MAXY)
            if q4 is not None:
                m &= (q4 == 1)
            if degr is not None:
                m &= (degr == 0)

            idx = np.where(m)[0]
            for i in idx:
                rows.append({
                    "beam": beam,
                    "agbd": float(agbd[i]),
                    "lat": float(lat[i]),
                    "lon": float(lon[i]),
                    "l4_quality_flag": int(q4[i]) if q4 is not None else None,
                    "degrade_flag": int(degr[i]) if degr is not None else None,
                    "file": str(h5_path),
                })

    df = pd.DataFrame(rows)
    if df.empty:
        return gpd.GeoDataFrame(df, geometry=[], crs="EPSG:4326")

    return gpd.GeoDataFrame(
        df,
        geometry=[Point(xy) for xy in zip(df["lon"], df["lat"])],
        crs="EPSG:4326"
    )

shots_list = []
for fp in downloaded:
    gdf = extract_l4a_agbd_points_aoi(fp)
    print(fp, "AOI shots:", len(gdf))
    if len(gdf) > 0:
        shots_list.append(gdf)

gedi_aoi = gpd.GeoDataFrame(pd.concat(shots_list, ignore_index=True), crs="EPSG:4326") if shots_list else gpd.GeoDataFrame(columns=["agbd"], geometry=[], crs="EPSG:4326")
print("Total AOI shots:", len(gedi_aoi))
gedi_aoi.head()


gedi_l4a_agbd/GEDI04_A_2019110092939_O01996_01_T03334_02_002_02_V002.h5 AOI shots: 0
gedi_l4a_agbd/GEDI04_A_2019137101716_O02416_04_T03984_02_002_02_V002.h5 AOI shots: 0
gedi_l4a_agbd/GEDI04_A_2019144073502_O02523_04_T02408_02_002_02_V002.h5 AOI shots: 2003
gedi_l4a_agbd/GEDI04_A_2019163122106_O02821_01_T02523_02_002_02_V002.h5 AOI shots: 390
gedi_l4a_agbd/GEDI04_A_2019169212519_O02920_04_T05407_02_002_02_V002.h5 AOI shots: 9826
gedi_l4a_agbd/GEDI04_A_2019174080623_O02989_01_T02064_02_002_02_V002.h5 AOI shots: 2449
gedi_l4a_agbd/GEDI04_A_2019195111704_O03317_04_T03831_02_002_02_V002.h5 AOI shots: 4797
gedi_l4a_agbd/GEDI04_A_2019214160606_O03615_01_T03487_02_002_02_V002.h5 AOI shots: 0
gedi_l4a_agbd/GEDI04_A_2019229101356_O03844_01_T02370_02_002_02_V002.h5 AOI shots: 213
gedi_l4a_agbd/GEDI04_A_2019248025112_O04134_01_T00641_02_002_02_V002.h5 AOI shots: 0
Total AOI shots: 19678


,beam,agbd,lat,lon,l4_quality_flag,degrade_flag,file,geometry
0,BEAM0000,102.984283,-3.713737,-70.540035,1,0,gedi_l4a_agbd/GEDI04_A_2019144073502_O02523_04...,POINT (-70.54004 -3.71374)
1,BEAM0000,181.824799,-3.714160,-70.539733,1,0,gedi_l4a_agbd/GEDI04_A_2019144073502_O02523_04...,POINT (-70.53973 -3.71416)
2,BEAM0000,154.345688,-3.714583,-70.539433,1,0,gedi_l4a_agbd/GEDI04_A_2019144073502_O02523_04...,POINT (-70.53943 -3.71458)
3,BEAM0000,101.812408,-3.715001,-70.539138,1,0,gedi_l4a_agbd/GEDI04_A_2019144073502_O02523_04...,POINT (-70.53914 -3.715)
4,BEAM0000,147.984161,-3.715424,-70.538836,1,0,gedi_l4a_agbd/GEDI04_A_2019144073502_O02523_04...,POINT (-70.53884 -3.71542)


## Site-level AGBD summary (optional buffer)  
We join GEDI shots to site geometries.  
- If `SITE_BUFFER_M = 0`, the join is strict (shots must fall within the site polygon).  
- If `SITE_BUFFER_M > 0`, we buffer sites in meters first (useful when sites are small and GEDI footprints are sparse).

In [15]:
# Buffer sites if requested
if SITE_BUFFER_M and SITE_BUFFER_M > 0:
    sites_m = sites_gdf.to_crs("EPSG:3857")
    sites_m["geometry"] = sites_m.geometry.buffer(SITE_BUFFER_M)
    sites_join = sites_m.to_crs("EPSG:4326")
else:
    sites_join = sites_gdf

joined = gpd.sjoin(
    gedi_aoi,
    sites_join[["site_id", "geometry"]],
    predicate="within",
    how="inner"
)

print("Shots matched to sites:", len(joined))

site_agbd = (
    joined.groupby("site_id")["agbd"]
    .agg(
        n="count",
        mean="mean",
        median="median",
        std="std",
        p10=lambda x: x.quantile(0.10),
        p90=lambda x: x.quantile(0.90),
        min="min",
        max="max",
    )
    .reset_index()
)

site_agbd["buffer_m"] = SITE_BUFFER_M
site_agbd


Shots matched to sites: 19678


,site_id,n,mean,median,std,p10,p90,min,max,buffer_m
0,site_1,19678,229.596929,203.909927,178.791097,68.343047,375.726318,0.0,5391.293945,50000


## Export results  
Writes the per-site AGBD summary table to CSV.

In [16]:
out_csv = "gedi_agbd_by_site.csv" if not SITE_BUFFER_M else f"gedi_agbd_by_site_buffer_{int(SITE_BUFFER_M)}m.csv"
site_agbd.to_csv(out_csv, index=False)
print("Wrote:", out_csv)

Wrote: gedi_agbd_by_site_buffer_50000m.csv
